# 04 — Evaluation

**Auralytics** | CPEN 355 Final Project

Evaluates trained checkpoints against the full test set (normal + anomalous).

This notebook produces:
- AUC-ROC, pAUC, F1 per machine type
- Score distribution plots (normal vs anomalous)
- ROC curves
- Reconstruction visualisations for correct and failed predictions
- A summary table ready to paste into the final report

**Requires:** trained `.pth` checkpoints in `models/` and processed data in `data/processed/`.

In [ ]:
# ── Portable repo-root resolution ────────────────────────────────────────────
import sys
from pathlib import Path

def _find_repo_root(marker: str = 'src') -> Path:
    candidate = Path.cwd()
    for _ in range(6):
        if (candidate / marker).is_dir():
            return candidate
        candidate = candidate.parent
    raise RuntimeError(f"Could not find repo root ('{marker}/' not found).")

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'Repo root: {REPO_ROOT}')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.dataset import DCASEDataset
from src.model import ConvAutoencoder
from src.evaluate import evaluate_machine, collect_scores, print_results_table
from src.utils import get_device, load_checkpoint, plot_reconstruction

plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e8e8e8',
    'axes.labelcolor':  '#e8e8e8',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
})

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
MODELS_DIR    = REPO_ROOT / 'models'
RESULTS_DIR   = REPO_ROOT / 'results'
MACHINE_TYPES = ['fan', 'pump', 'valve']

device = get_device()
print(f'PyTorch {torch.__version__} | device: {device}')

## 1. Run Evaluation — All Machine Types

Skips any machine without a checkpoint — safe to run even if only fan is trained so far.

In [ ]:
all_results = []

for machine in MACHINE_TYPES:
    ckpt = MODELS_DIR / f'{machine}_best.pth'
    if not ckpt.exists():
        print(f'[skip] {machine} — no checkpoint found at {ckpt}')
        continue
    print(f'\nEvaluating {machine.upper()}...')
    result = evaluate_machine(
        machine_type  = machine,
        processed_dir = PROCESSED_DIR,
        models_dir    = MODELS_DIR,
        results_dir   = RESULTS_DIR,
        show_plots    = False,    # saved to results/ — displayed below
    )
    all_results.append(result)
    print(f'  AUC={result["auc"]:.3f}  pAUC={result["pauc"]:.3f}  F1={result["f1"]:.3f}')

## 2. Summary Table
Copy this into your final report.

In [ ]:
if all_results:
    print_results_table(all_results)
else:
    print('No results yet — train at least one machine type first.')

## 3. Score Distributions
Good separation = the model is learning. Overlap = hard cases.

In [ ]:
for result in all_results:
    img_path = RESULTS_DIR / f"{result['machine']}_score_dist.png"
    if img_path.exists():
        img = plt.imread(str(img_path))
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(result['machine'].upper(), fontweight='bold')
        plt.tight_layout()
        plt.show()

## 4. ROC Curves

In [ ]:
for result in all_results:
    img_path = RESULTS_DIR / f"{result['machine']}_roc.png"
    if img_path.exists():
        img = plt.imread(str(img_path))
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(img)
        ax.axis('off')
        plt.tight_layout()
        plt.show()

## 5. Reconstruction Deep-Dive

Pick a machine and look at what the model actually does to normal vs anomalous clips.
Good failure analysis material for the report.

In [ ]:
MACHINE = 'fan'   # change to any trained machine type

model = ConvAutoencoder(base_ch=32).to(device)
load_checkpoint(MODELS_DIR / f'{MACHINE}_best.pth', model, device=device)

test_ds = DCASEDataset(PROCESSED_DIR, MACHINE, split='test')
loader  = DataLoader(test_ds, batch_size=16, shuffle=True, num_workers=0)

specs, labels = next(iter(loader))
specs = specs.to(device)

with torch.no_grad():
    recons = model(specs)
    scores = model.anomaly_score(specs)

print(f'Batch scores — normal   : {scores[labels==0].mean():.4f} mean')
print(f'Batch scores — anomalous: {scores[labels==1].mean():.4f} mean')

In [ ]:
# Look up threshold for the specific MACHINE selected above
result = next((r for r in all_results if r["machine"] == MACHINE), None)
if result is None:
    print(f"[warn] No evaluation result found for {MACHINE}. Run Section 1 first, or set threshold manually.")
    threshold = 0.15   # fallback only
else:
    threshold = result["threshold"]
    print(f"Using threshold {threshold:.5f} from {MACHINE} evaluation")

for target_label, name in [(0, "Normal"), (1, "Anomalous")]:
    idx = next((i for i, l in enumerate(labels) if l == target_label), None)
    if idx is None:
        print(f"No {name} clip in this batch — re-run cell")
        continue
    score = scores[idx].item()
    pred  = "ANOMALOUS" if score >= threshold else "NORMAL"
    truth = "ANOMALOUS" if target_label == 1 else "NORMAL"
    correct = "CORRECT" if pred == truth else "WRONG"
    print(f"\n{name} clip — score: {score:.4f} | pred: {pred} | truth: {truth} | {correct}")
    plot_reconstruction(
        original      = specs[idx].cpu().numpy(),
        reconstructed = recons[idx].cpu().numpy(),
        anomaly_score = score,
        label         = target_label,
    )


## 6. Failure Case Analysis

Find the hardest clips — highest-scoring normals and lowest-scoring anomalies.
These are the false positives and false negatives worth visualising in the report.

In [ ]:
# Collect full test scores for failure analysis
full_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

from src.evaluate import collect_scores
all_scores, all_labels = collect_scores(model, full_loader, device)

normal_scores    = all_scores[all_labels == 0]
anomalous_scores = all_scores[all_labels == 1]

print(f'Normal    — mean: {normal_scores.mean():.4f}  max: {normal_scores.max():.4f}  min: {normal_scores.min():.4f}')
print(f'Anomalous — mean: {anomalous_scores.mean():.4f}  max: {anomalous_scores.max():.4f}  min: {anomalous_scores.min():.4f}')

overlap = (normal_scores.max() > anomalous_scores.min())
print(f'\nScore ranges overlap: {overlap}  (overlap = false positives/negatives exist at any threshold)')

## ✅ Evaluation Complete

Outputs saved to `results/`:
- `{machine}_score_dist.png` — score distributions with threshold line
- `{machine}_roc.png`        — ROC curve with pAUC boundary

**Next:** `backend/` — wrap the model in FastAPI for the live demo.